# Unit 5 — Reference Solution: Train a T-GCN on the Bus Corridor, Then Ask If It Earned Its Complexity

> ## Disclaimer — AI-generated reference, provided for learning
>
> **This solution notebook was generated by the course's AI assistant** (with
> full access to the unit's research synthesis, the Unit 5 demo notebook + its
> helpers, the Unit 3 breakeven result, and the practice task). It is shared with
> you **from day one, on purpose**: a complete, runnable solution shows the
> practice task is *feasible* and gives you a concrete reference for the *shape*
> of a strong direct → interpret → extend answer.
>
> **Two honest caveats:**
> 1. **It has not been fully Run-All'd end-to-end on Colab yet.** It passed static
>    checks (valid notebook JSON; every code cell compiles) and reuses the demo's
>    already-designed helpers verbatim. The PyG install, the T-GCN/A3T-GCN
>    training, and folium still need a human **Colab Run-All** for full parity.
>    Treat it as a reference, not a guaranteed-runnable artifact — if a cell
>    breaks, fixing it is good practice.
> 2. **The task is open-ended by design — there is no single correct answer.**
>    This is *one* defensible path (undirected baseline T-GCN → breakeven vs U3
>    SARIMA → attention/cross-correlation honesty → directed-vs-undirected →
>    honest overkill verdict). Your own adjacency choice, window, horizons, and
>    zero-speed handling can be just as correct. Do **not** copy this — use it to
>    check your reasoning. You are graded on your own direct → interpret → extend
>    loop, not on matching this notebook (and an honest "the GNN is overkill here"
>    is a top answer).

**The synthesis.** Unit 3 gave you a **breakeven horizon** per corridor segment
with SARIMA. The Unit 5 demo showed a **T-GCN** (GCN + GRU) close the **spatial
gap** on METR-LA, and showed that A3T-GCN's attention is **temporal** (over past
time steps), *not* spatial (over nodes). This notebook takes the **same
~10-segment corridor** re-framed as a graph and asks: **did the GNN push the
breakeven outward vs your U3 SARIMA — or did you overengineer a ~10-node/17-day
problem?**

**Stack:** the demo's helpers (`fetch_with_fallback`, `masked_metrics`,
`plot_error_distributions`, `plot_loss_curve`, `TGCNForecaster`) + `torch` /
`torch_geometric_temporal`. **No new heavy dependencies.**

**What it covers:** the baseline (bus graph → adjacency choice → small T-GCN →
per-horizon eval → breakeven vs a floor + U3 SARIMA) and the extensions:
**(a)** attention at a congestion event *with the temporal-≠-spatial honesty
guardrail*; **(b)** interrogate the worst-predicted segment; **(c)** the
directed-vs-undirected design decision + the honest "is a GNN overkill here?"
verdict.


## Setup — fetch the course helper and install this unit's requirements

In [ ]:
# --- Setup: fetch the course helper and install this unit's requirements -----
# On Colab: pip-installs requirements/unit-5.txt (no torch pin; scatter/sparse
# intentionally omitted). Locally: no-op (env already set up with `uv`).
import os, sys, subprocess, urllib.request

SETUP_URL = (
    "https://raw.githubusercontent.com/"
    "bgalon/geo-graph-learning/main/setup_colab.py"
)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("setup_colab.py"):
        urllib.request.urlretrieve(SETUP_URL, "setup_colab.py")
    import setup_colab
    setup_colab.setup_unit("unit-5")
else:
    print("Local environment detected — skipping Colab install "
          "(assuming `uv sync --extra unit-5`).")


## Smoke test — fail fast, fail clean

In [ ]:
# --- Smoke test --------------------------------------------------------------
# Capture any failure, then re-raise OUTSIDE the except block (IPython-9 safe).
_smoke_error = None
try:
    import torch
    import torch_geometric
    from torch_geometric.nn import GCNConv
    import torch_geometric_temporal
    from torch_geometric_temporal.nn.recurrent import TGCN, A3TGCN
    import sklearn
    from sklearn.metrics import mean_absolute_error
    import matplotlib
    import matplotlib.pyplot as plt
    import folium
    import numpy as np

    _ = torch.tensor([1.0, 2.0]).mean().item()
    _ = GCNConv(2, 2)
    _ = mean_absolute_error([1.0, 2.0], [1.1, 1.9])
    _ = folium.Map(location=[32.08, 34.78], zoom_start=13)
    _fig = plt.figure(); plt.close(_fig)
except Exception as exc:                                # noqa: BLE001
    _smoke_error = exc

if _smoke_error is not None:
    print("=" * 64)
    print("SMOKE TEST FAILED — environment is not ready.")
    print(f"  {type(_smoke_error).__name__}: {_smoke_error}")
    print("  See unit-5-gnn/KNOWN_ISSUES.md (torch/PyG version conflicts).")
    print("=" * 64)
    raise _smoke_error from None

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("\u2713 smoke test passed")
print(f"  torch            {torch.__version__}")
print(f"  torch_geometric  {torch_geometric.__version__}")
print(f"  PyG-Temporal     {torch_geometric_temporal.__version__}")
print(f"  device           {_DEVICE.upper()}  "
      "(the bus graph is TINY — CPU is fine, seconds per model)")


## Helpers — reused verbatim from the demo (+ two bus-specific tweaks)

The demo already gave us `fetch_with_fallback`, `masked_metrics`,
`plot_error_distributions`, `plot_loss_curve`, and the `TGCNForecaster` class.
We reuse them and add **two bus-specific things**:

- **`bus_metrics`** — like `masked_metrics`, **but it does NOT mask zeros.** On
  METR-LA `0 = missing`; on the **bus a `0` means STOPPED** — a real low speed
  and often the very congestion we forecast. Masking it would delete the signal.
  (MAPE still skips exact-zero *true* speeds to avoid dividing by zero.)
- **`make_windows`** — the loader the practice owns: slice the `[segments, time]`
  speed matrix into `(input-window, multi-horizon-target)` pairs.


In [ ]:
# --- Config + fetch_with_fallback (verbatim from the demo helper cell) --------
import os, numpy as np
CACHE_DIR = "/content" if IN_COLAB else "."

# Tiny pre-built bus graph + a short cached speed slice (the Unit 3 corridor).
BUS = {"gdrive_id": "<GDRIVE_ID_bus_graph>",
       "http": "https://github.com/bgalon/geo-graph-learning/releases/download/u5-data/bus_corridor_u5.npz"}

def fetch_with_fallback(dest_name, gdrive_id, http_url, label):
    """Return a local cached path: try gdown (Drive) if a real id is set, else the
    HTTP (GitHub Release) mirror. Caches to CACHE_DIR. (Verbatim from the demo.)"""
    dest = os.path.join(CACHE_DIR, dest_name)
    if os.path.exists(dest):
        print(f"\u2713 {label}: using cached {dest}")
        return dest
    _has_gid = bool(gdrive_id) and not str(gdrive_id).startswith("<")
    if _has_gid:
        try:
            import gdown
            gdown.download(id=gdrive_id, output=dest, quiet=True)
            if os.path.exists(dest) and os.path.getsize(dest) > 0:
                print(f"\u2713 {label}: downloaded via gdown (Drive) -> {dest}")
                return dest
            raise RuntimeError("gdown produced no file")
        except Exception as exc:                          # noqa: BLE001
            print(f"\u26a0 {label}: gdown failed ({exc}); FALLING BACK to HTTP mirror")
    try:
        import urllib.request
        urllib.request.urlretrieve(http_url, dest)
        print(f"\u2713 {label}: downloaded via HTTP (release) -> {dest}")
        return dest
    except Exception as exc:                              # noqa: BLE001
        raise RuntimeError(f"{label}: fetch failed ({exc}).") from None


In [ ]:
# --- Metrics + plot helpers (demo helpers; bus_metrics is the zero-speed tweak)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def bus_metrics(y_true, y_pred):
    """MAE / RMSE over ALL bins (a stopped-bus 0 is a REAL low speed, not missing —
    the OPPOSITE of METR-LA). MAPE skips only exact-zero *true* speeds (div-by-0)."""
    yt = np.asarray(y_true, float).ravel()
    yp = np.asarray(y_pred, float).ravel()
    mae = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    m = yt > 1e-6
    mape = float(np.mean(np.abs((yt[m] - yp[m]) / yt[m])) * 100.0) if m.sum() else float("nan")
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

def plot_error_distributions(errors_by_model, horizon_label, units="km/h"):
    """box + histogram + ECDF side by side (verbatim shape from the demo)."""
    names = list(errors_by_model)
    data = [np.asarray(errors_by_model[n], float).ravel() for n in names]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    try:
        axes[0].boxplot(data, tick_labels=names, showfliers=False)
    except TypeError:
        axes[0].boxplot(data, labels=names, showfliers=False)
    axes[0].set(title=f"Abs error spread @ {horizon_label}", ylabel=f"|error| ({units})")
    for n, d in zip(names, data):
        axes[1].hist(d, bins=30, alpha=0.5, label=n, density=True)
    axes[1].set(title="Error histogram", xlabel=f"|error| ({units})", ylabel="density")
    axes[1].legend()
    for n, d in zip(names, data):
        ds = np.sort(d); ecdf = np.arange(1, len(ds) + 1) / len(ds)
        axes[2].plot(ds, ecdf, label=n)
    axes[2].set(title="Error ECDF (more left/up = more small errors)",
                xlabel=f"|error| ({units})", ylabel="cumulative fraction")
    axes[2].legend(); fig.tight_layout()
    return fig

def plot_loss_curve(losses, title="Training loss"):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(range(1, len(losses) + 1), losses, marker="o")
    ax.set(title=title, xlabel="epoch", ylabel="MSE loss"); fig.tight_layout()
    return fig


In [ ]:
# --- TGCNForecaster: reused VERBATIM from the demo (handles 2-D or 3-D input) --
import torch
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import TGCN

class TGCNForecaster(torch.nn.Module):
    """T-GCN encoder + linear head mapping hidden state -> horizon.
    x is [nodes, features, periods]; the GRU consumes the periods one step at a
    time (space via the GCN, then time). A 2-D [nodes, features] input is one step."""
    def __init__(self, node_features, hidden, horizon):
        super().__init__()
        self.recurrent = TGCN(in_channels=node_features, out_channels=hidden)
        self.head = torch.nn.Linear(hidden, horizon)

    def forward(self, x, edge_index, edge_weight=None):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        H = None
        for step in range(x.shape[-1]):
            H = self.recurrent(x[:, :, step], edge_index, edge_weight, H)
        return self.head(F.relu(H))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


## Load the bus corridor as a graph — and CHOOSE the adjacency (graded decision)

We fetch `bus_corridor_u5.npz` exactly as the demo's Section-8 cell does:
`edge_index` (the directed segment chain), `speeds [segments, time]` (the U3
per-segment speed series on 15-min bins), `latlon [segments, 2]`. If the asset
isn't hosted yet, we fall back to a **synthetic Ibn-Gabirol-shaped corridor** (a
congestion wave sweeping downstream) so this notebook is self-contained.

**The graded decision.** The raw chain is *directed* (segment `i` feeds `i+1`).
We build **both** adjacencies and use **undirected symmetric-normalized** as the
T-GCN-default baseline (extension c revisits directed):

- **Undirected** — `i` and `i+1` exchange messages both ways; what a symmetric
  GCN wants, but lets a downstream slowdown leak upstream (non-physical).
- **Directed** — upstream → downstream only; matches the physics + U3's lagged
  cross-correlation, but a symmetric-normalized GCN isn't built for it.


In [ ]:
# --- Load the corridor graph (demo fetch idiom) with a synthetic fallback ------
try:
    bus_path = fetch_with_fallback("bus_corridor_u5.npz", BUS["gdrive_id"],
                                   BUS["http"], label="bus corridor graph")
    busz = np.load(bus_path, allow_pickle=True)
    chain_edges = np.asarray(busz["edge_index"])       # [2, E], directed i->i+1
    speeds = np.asarray(busz["speeds"], dtype=float)   # [segments, time], km/h, 15-min bins
    latlon = np.asarray(busz["latlon"], dtype=float)   # [segments, 2]
    IS_REAL = True
    print(f"\u2713 bus graph: {speeds.shape[0]} segment-nodes, {speeds.shape[1]} time bins (REAL corridor)")
except Exception as exc:                               # noqa: BLE001
    IS_REAL = False
    print(f"\u2139 bus graph not hosted yet — SYNTHETIC Ibn-Gabirol-shaped fallback [{type(exc).__name__}]")
    latlon = np.array([
        (32.07104, 34.78200), (32.07360, 34.78175), (32.07549, 34.78173),
        (32.07816, 34.78121), (32.07954, 34.78125), (32.08155, 34.78127),
        (32.08453, 34.78157), (32.08809, 34.78238), (32.09006, 34.78275),
        (32.09180, 34.78292)])
    N = len(latlon); T = 900                              # ~17 days x ~53 bins/day-ish, coarse
    chain_edges = np.array([list(range(N - 1)), list(range(1, N))])
    rng = np.random.default_rng(5)
    # a daily-dominated series (the reason a GNN may be OVERKILL) + a downstream-sweeping wave
    tod = np.sin(np.linspace(0, 2 * np.pi * (T / 53.0), T))       # crude daily cycle
    speeds = np.empty((N, T))
    for s in range(N):
        base = 26 + 4 * tod                               # each segment ~ its own daily pattern
        noise = rng.normal(0, 2.5, T)
        speeds[s] = base + noise
    for onset in range(80, T, 160):                       # recurring rush waves, later downstream
        for s in range(N):
            a = onset + s * 2
            speeds[s, a:a + 8] = rng.normal(8, 2, size=min(8, max(0, T - a)))
    speeds = np.abs(speeds).clip(0, 45)
    speeds[:, ::120] = 0.0                                 # a few STOPPED bins (velocity==0, REAL)

N_SEG, T_BINS = speeds.shape

# Build BOTH adjacencies. Baseline = undirected (symmetric); directed used in ext (c).
undirected_ei = torch.tensor(
    np.concatenate([chain_edges, chain_edges[::-1]], axis=1), dtype=torch.long)
directed_ei = torch.tensor(chain_edges, dtype=torch.long)
print(f"undirected edges: {undirected_ei.shape[1]}   directed edges: {directed_ei.shape[1]}")

# Target = a downstream segment; upstream = its immediate predecessor.
TARGET_NODE = N_SEG - 3
UPSTREAM_NODE = TARGET_NODE - 1
print(f"target segment-node {TARGET_NODE}, upstream neighbour {UPSTREAM_NODE}")


In [ ]:
# --- Draw the corridor: nodes = segments, coloured by a mid-window speed --------
import folium
snap_t = min(150, T_BINS - 1)
sv = speeds[:, snap_t]; vmin, vmax = float(sv.min()), float(sv.max())
def _bcol(v):
    f = 0.0 if vmax == vmin else max(0.0, min(1.0, (v - vmin) / (vmax - vmin)))
    return f"#{int(255*(1-f)):02x}{int(255*f):02x}33"
bm = folium.Map(location=list(latlon.mean(axis=0)), zoom_start=14, tiles="CartoDB positron")
for a, b in directed_ei.t().tolist():
    folium.PolyLine([list(latlon[a]), list(latlon[b])], color="#555", weight=3, opacity=0.7).add_to(bm)
for i, (la, lo_) in enumerate(latlon):
    folium.CircleMarker([la, lo_], radius=8, color="black", weight=1, fill=True,
                        fill_color=_bcol(float(sv[i])), fill_opacity=0.9,
                        popup=f"segment {i} · speed {sv[i]:.0f} km/h").add_to(bm)
print("corridor drawn (red = slow). nodes = segments, edges = adjacency.")
bm


## The loader the practice owns: window the speed matrix

`make_windows` slices `[segments, time]` into `(input-window, target)` pairs. The
input window ends at bin `t`; the target is bins `t+1 … t+H_max` (no overlap —
overlapping would leak the answer). Horizons **15/30/60 min** on 15-min bins map
to future-step indices **0 / 1 / 3** — the same convention as your U3 SARIMA
result. We split windows **chronologically** (first 80% train).


In [ ]:
# --- make_windows: (input-window, multi-horizon target) pairs ------------------
NUM_IN = 8            # 8 past 15-min bins (2 h) the GRU sees
NUM_OUT = 4           # predict 4 future bins -> covers 15/30/45/60 min
HORIZON_STEPS = {"15 min": 0, "30 min": 1, "60 min": 3}   # future-step index (0-based)
HORIZON_MIN = {"15 min": 15, "30 min": 30, "60 min": 60}

def make_windows(spd, num_in, num_out):
    """spd: [N, T] -> lists X (each [N, num_in]) and Y (each [N, num_out])."""
    N, T = spd.shape
    X, Y = [], []
    for t in range(num_in, T - num_out + 1):
        X.append(spd[:, t - num_in:t])          # [N, num_in]
        Y.append(spd[:, t:t + num_out])         # [N, num_out]
    return X, Y

X_all, Y_all = make_windows(speeds, NUM_IN, NUM_OUT)
split = int(0.8 * len(X_all))
X_tr, Y_tr = X_all[:split], Y_all[:split]
X_te, Y_te = X_all[split:], Y_all[split:]
print(f"{len(X_all)} windows total -> {len(X_tr)} train / {len(X_te)} test")

# Zero-speed decision (the velocity==0 trap): we KEEP stopped bins as real low
# speed (do NOT mask). Documented in bus_metrics.
train_speeds = speeds[:, :split + NUM_IN]      # rough train slice for the floor
print("zero-speed policy: stops kept as REAL low speed (not masked as missing).")


## Train a small T-GCN on the corridor (undirected baseline)

In [ ]:
# --- train_tgcn: mirror the demo's loop (Adam, MSE, step the GRU) --------------
def train_tgcn(X, Y, edge_index, edge_weight=None, hidden=16, epochs=40, horizon=NUM_OUT, lr=1e-2):
    model = TGCNForecaster(node_features=1, hidden=hidden, horizon=horizon).to(device)
    ei = edge_index.to(device)
    ew = edge_weight.to(device) if edge_weight is not None else None
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for ep in range(epochs):
        model.train(); tot = 0.0; n = 0
        for x, y in zip(X, Y):
            xt = torch.as_tensor(x, dtype=torch.float).unsqueeze(1).to(device)  # [N,1,num_in]
            yt = torch.as_tensor(y, dtype=torch.float).to(device)               # [N, horizon]
            opt.zero_grad()
            yhat = model(xt, ei, ew)
            loss = F.mse_loss(yhat, yt)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        losses.append(tot / max(n, 1))
    return model, losses

model_u, losses_u = train_tgcn(X_tr, Y_tr, undirected_ei, epochs=40)
print(f"\u2713 trained undirected T-GCN. final loss {losses_u[-1]:.3f}")
plot_loss_curve(losses_u, title="Bus T-GCN (undirected) — training loss")


## Evaluate at 15/30/60 min vs a floor + your U3 SARIMA — the breakeven read

For the **target segment** we compute per-horizon error and compare it to:
1. a **historical-average floor** — predict the target's *training-mean* speed
   (constant), the dumb baseline every model must beat; and
2. **your U3 SARIMA** per-horizon numbers for this segment (hard-coded here as a
   **PLACEHOLDER** — substitute your own U3 result).

The **breakeven horizon** is the last horizon where the T-GCN still beats the
floor. The capstone question: did it push breakeven **outward** vs U3?


In [ ]:
# --- Per-horizon eval on the target node ---------------------------------------
def eval_horizons(model, X, Y, edge_index, target_node, edge_weight=None):
    model.eval(); ei = edge_index.to(device)
    ew = edge_weight.to(device) if edge_weight is not None else None
    preds = {h: [] for h in HORIZON_STEPS}; acts = {h: [] for h in HORIZON_STEPS}
    with torch.no_grad():
        for x, y in zip(X, Y):
            xt = torch.as_tensor(x, dtype=torch.float).unsqueeze(1).to(device)
            yhat = model(xt, ei, ew).cpu().numpy()
            for h, step in HORIZON_STEPS.items():
                preds[h].append(float(yhat[target_node, step]))
                acts[h].append(float(y[target_node, step]))
    return preds, acts

preds_u, acts_u = eval_horizons(model_u, X_te, Y_te, undirected_ei, TARGET_NODE)

# T-GCN per-horizon RMSE + the floor (constant training-mean of the target node)
floor_val = float(train_speeds[TARGET_NODE].mean())
tgcn_rmse, floor_rmse, tgcn_err = {}, {}, {}
for h in HORIZON_STEPS:
    a = np.array(acts_u[h]); p = np.array(preds_u[h])
    tgcn_rmse[h] = bus_metrics(a, p)["RMSE"]
    floor_rmse[h] = float(np.sqrt(np.mean((a - floor_val) ** 2)))
    tgcn_err[h] = np.abs(a - p)

# --- PLACEHOLDER: your U3 SARIMA per-horizon RMSE for THIS segment -------------
# Replace with your own Unit-3 numbers. Representative values shown so the plot renders.
u3_sarima_rmse = {"15 min": 3.4, "30 min": 4.6, "60 min": 6.1}

print("target-node per-horizon RMSE (km/h):")
for h in HORIZON_STEPS:
    print(f"  {h}: T-GCN {tgcn_rmse[h]:.2f} | floor {floor_rmse[h]:.2f} | U3 SARIMA {u3_sarima_rmse[h]:.2f}")


In [ ]:
# --- Breakeven plot: T-GCN vs floor vs U3 SARIMA -------------------------------
xs = [HORIZON_MIN[h] for h in HORIZON_STEPS]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(xs, [tgcn_rmse[h] for h in HORIZON_STEPS], "o-", label="T-GCN (undirected)")
ax.plot(xs, [u3_sarima_rmse[h] for h in HORIZON_STEPS], "s--", label="U3 SARIMA")
ax.plot(xs, [floor_rmse[h] for h in HORIZON_STEPS], "d:", color="gray", label="historical-average floor")

# breakeven = last horizon where T-GCN still beats the floor
beats = [HORIZON_MIN[h] for h in HORIZON_STEPS if tgcn_rmse[h] < floor_rmse[h]]
breakeven = max(beats) if beats else None
if breakeven is not None:
    ax.axvline(breakeven, color="crimson", ls="--", label=f"T-GCN breakeven ~{breakeven} min")
ax.set(title="Did the GNN push the breakeven outward? (target segment)",
       xlabel="forecast horizon (min)", ylabel="RMSE (km/h)")
ax.legend(); fig.tight_layout()
print(f"T-GCN breakeven horizon: {breakeven} min "
      + ("(beats the floor out to here)" if breakeven else "(never beats the floor — overkill signal!)"))
# INTERPRET: compare `breakeven` to your U3 SARIMA breakeven. On a ~10-node/17-day
# corridor the gain is often small and shrinks past 30 min — that is on-thesis.
plot_error_distributions({"T-GCN": tgcn_err["30 min"],
                          "|floor err|": np.abs(np.array(acts_u["30 min"]) - floor_val)},
                         "30 min")


## Extension (a) — attention at a congestion event, with the honesty guardrail

We train a small **A3T-GCN** and read its **temporal attention** for the target
segment: weight vs **past input step**. Then — because that is **temporal, not
spatial** — we put U3's **lagged upstream→downstream cross-correlation** on a
**separate axis** to answer "which upstream node?".

> **HARD guardrail (Face c):** a temporal-attention bar is **NOT** spatial
> causality. The attention says *which past moments* mattered; the
> cross-correlation says *which upstream segment leads*. Two axes, never merged.


In [ ]:
# --- Train a tiny A3T-GCN (attention over the NUM_IN past steps) ---------------
from torch_geometric_temporal.nn.recurrent import A3TGCN

class A3TGCNForecaster(torch.nn.Module):
    def __init__(self, node_features, hidden, periods, horizon):
        super().__init__()
        self.recurrent = A3TGCN(in_channels=node_features, out_channels=hidden, periods=periods)
        self.head = torch.nn.Linear(hidden, horizon)
    def forward(self, x, edge_index, edge_weight=None):
        # x: [nodes, features, periods]
        h = self.recurrent(x, edge_index, edge_weight)
        return self.head(F.relu(h))

a3t = A3TGCNForecaster(node_features=1, hidden=16, periods=NUM_IN, horizon=NUM_OUT).to(device)
_ei = undirected_ei.to(device)
opt = torch.optim.Adam(a3t.parameters(), lr=1e-2)
for ep in range(30):
    a3t.train(); tot = 0.0; n = 0
    for x, y in zip(X_tr, Y_tr):
        xt = torch.as_tensor(x, dtype=torch.float).unsqueeze(1).to(device)  # [N,1,periods]
        yt = torch.as_tensor(y, dtype=torch.float).to(device)
        opt.zero_grad(); loss = F.mse_loss(a3t(xt, _ei), yt)
        loss.backward(); opt.step(); tot += loss.item(); n += 1
print(f"\u2713 trained A3T-GCN. final loss {tot / max(n,1):.3f}")

# TEMPORAL attention: A3TGCN's global per-step weighting (softmax over periods)
with torch.no_grad():
    attn = torch.nn.functional.softmax(a3t.recurrent._attention, dim=0).cpu().numpy().ravel()
print(f"temporal attention over {len(attn)} past steps (sums to 1): {np.round(attn, 3)}")


In [ ]:
# --- TWO SEPARATE AXES: temporal attention (top) + U3 cross-correlation (bottom)
us = speeds[UPSTREAM_NODE]; dsx = speeds[TARGET_NODE]
maxlag = 6; ccf = []
for k in range(maxlag + 1):
    a = us[:len(us) - k]; b = dsx[k:]
    m = np.isfinite(a) & np.isfinite(b)
    ccf.append(np.corrcoef(a[m], b[m])[0, 1] if m.sum() > 2 else np.nan)
ccf = np.array(ccf); peak_lag = int(np.nanargmax(ccf))

fig, (axT, axS) = plt.subplots(2, 1, figsize=(9, 7))
axT.bar(range(1, len(attn) + 1), attn, color="#3b6")
axT.set(title="TEMPORAL — A3T-GCN attention over PAST TIME STEPS (global, NOT spatial)",
        xlabel="past input step (1=oldest ... latest)", ylabel="attention weight")
axS.plot(range(maxlag + 1), ccf, "o-", color="#36b")
axS.axvline(peak_lag, color="crimson", ls="--", label=f"peak at lag {peak_lag} (~{peak_lag*15} min)")
axS.set(title="SPATIAL — U3 upstream->downstream cross-correlation (the 'which node' axis)",
        xlabel="lag k (bins): upstream leads downstream by k", ylabel="correlation")
axS.legend(); fig.tight_layout()
print("HONESTY CAVEAT: the top bar is TIME (which past step); the bottom is SPACE "
      "(which upstream segment leads). A temporal heatmap is NOT spatial causality.")


## Extension (b) — interrogate the WORST-predicted segment

Rank all segments by 15-min test RMSE, pick the worst, and ask **what the graph
is missing** there — is it a low-degree (dead-end) node with no neighbour for
message passing to borrow from, a thin-data segment, or genuine signal-timing
noise no neighbour predicts?


In [ ]:
# --- Worst-segment interrogation -----------------------------------------------
model_u.eval(); _ei = undirected_ei.to(device)
per_node_sq = np.zeros(N_SEG); cnt = 0
with torch.no_grad():
    for x, y in zip(X_te, Y_te):
        xt = torch.as_tensor(x, dtype=torch.float).unsqueeze(1).to(device)
        yhat = model_u(xt, _ei).cpu().numpy()
        step = HORIZON_STEPS["15 min"]
        per_node_sq += (yhat[:, step] - np.asarray(y)[:, step]) ** 2
        cnt += 1
per_node_rmse = np.sqrt(per_node_sq / max(cnt, 1))
worst = int(np.argmax(per_node_rmse))

# graph degree of each node (undirected)
deg = np.zeros(N_SEG, dtype=int)
for a, b in undirected_ei.t().tolist():
    deg[a] += 1
print("per-segment 15-min RMSE (km/h):")
for i in range(N_SEG):
    tag = "  <-- WORST" if i == worst else ""
    print(f"  seg {i}: RMSE {per_node_rmse[i]:.2f}  degree {deg[i]}{tag}")
print(f"\nworst segment = {worst} (degree {deg[worst]}).")
print("INTERPRET: is it a degree-1 endpoint (no neighbour to borrow from), a "
      "thin-data segment, or genuine junction noise? Name what data would help.")


## Extension (c) — directed vs undirected + the honest "is a GNN overkill here?"

We now train the **directed** (upstream→downstream) adjacency and compare its
target-segment breakeven to the undirected baseline and U3 SARIMA. Then we give
the **honest verdict**: on a ~10-node / ~17-day corridor whose series is
dominated by its own daily pattern, does the GNN *earn* its complexity — or is it
overengineered (STAEformer's "diminishing returns")? **An honest "overkill,
here's the measured reason" is a top answer.**


In [ ]:
# --- Directed adjacency: train + compare breakeven -----------------------------
model_d, losses_d = train_tgcn(X_tr, Y_tr, directed_ei, epochs=40)
preds_d, acts_d = eval_horizons(model_d, X_te, Y_te, directed_ei, TARGET_NODE)
d_rmse = {h: bus_metrics(np.array(acts_d[h]), np.array(preds_d[h]))["RMSE"] for h in HORIZON_STEPS}

print("target-node RMSE by horizon (km/h): directed vs undirected vs floor vs U3")
for h in HORIZON_STEPS:
    print(f"  {h}: directed {d_rmse[h]:.2f} | undirected {tgcn_rmse[h]:.2f} | "
          f"floor {floor_rmse[h]:.2f} | U3 {u3_sarima_rmse[h]:.2f}")

def _breakeven(rmse):
    b = [HORIZON_MIN[h] for h in HORIZON_STEPS if rmse[h] < floor_rmse[h]]
    return max(b) if b else None
be_u, be_d = _breakeven(tgcn_rmse), _breakeven(d_rmse)
print(f"\nbreakeven — undirected: {be_u} min | directed: {be_d} min")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(xs, [tgcn_rmse[h] for h in HORIZON_STEPS], "o-", label="T-GCN undirected")
ax.plot(xs, [d_rmse[h] for h in HORIZON_STEPS], "^-", label="T-GCN directed")
ax.plot(xs, [u3_sarima_rmse[h] for h in HORIZON_STEPS], "s--", label="U3 SARIMA")
ax.plot(xs, [floor_rmse[h] for h in HORIZON_STEPS], "d:", color="gray", label="floor")
ax.set(title="Adjacency design decision: directed vs undirected vs U3",
       xlabel="horizon (min)", ylabel="RMSE (km/h)"); ax.legend(); fig.tight_layout()


### The honest verdict (INTERPRET + EXTEND)

Fill this in from **your** numbers, not the reference's. The template of a strong
answer:

- **Did the GNN earn its complexity?** Compare the T-GCN breakeven to your U3
  SARIMA breakeven. If it moved out by a bin or two *and* the win is largest at
  short horizons (where the upstream signal arrives), the GNN earned it. If it
  barely moved — or the floor wins past 30 min — it did not.
- **Directed vs undirected:** which wired the corridor better for the downstream
  target, and does that match the one-way physics of bus flow?
- **Overkill condition (a top answer if honest):** a ~10-node graph over ~17
  days, with each segment dominated by its own daily cycle, is exactly where a
  GNN is *overengineered* — ten independent SARIMAs are cheaper and about as
  good. This is the field's own admission (STAEformer: "diminishing returns from
  architectural complexity"). Argue it **from your breakeven numbers**.


## Where to go next

1. **Add an upstream feature.** Give each node its upstream neighbour's recent
   speed as an extra node feature and re-check whether the downstream breakeven
   extends — the multivariate hypothesis U3's cross-correlation raised.
2. **The capstone end-game (homework).** Roll the trained T-GCN forward to predict
   the **A→B travel-time** at a future departure, and ask how far ahead a rider
   should trust it (your breakeven horizon is the answer).
3. **Go spatial-attention.** Swap A3T-GCN's temporal attention for a GAT-style
   spatial-attention layer and ask whether the "which node" story it tells matches
   the cross-correlation evidence — the honest interrogation, one level deeper.

---

<sub>© Geospatial Graph Learning — rights reserved (not open-CC); see the course
`NOTICE.md`. This notebook was generated with AI assistance and reviewed by the
instructor; static-checked (valid JSON + every code cell compiles), Colab Run-All
pending.</sub>
